In [0]:
from pyspark.sql.functions import col as spark_col, sum as spark_sum
from pyspark.sql import Row
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
{
    "check_name": "",
    "status": "",
    "src_count": None,
    "tgt_count": None,
    "mismatch_count": None,
    "duplicate_count": None,
    "null_count": None,
    "remarks": ""
}

In [0]:
def row_count_check(spark, src_table, tgt_table):
    src_count = spark.table(src_table).count()
    tgt_count = spark.table(tgt_table).count()

    return {
        "check_name": "row_count_check",
        "status": "PASS" if src_count == tgt_count else "FAIL",
        "src_count": src_count,
        "tgt_count": tgt_count,
        "mismatch_count": abs(src_count - tgt_count),
        "duplicate_count": None,
        "null_count": None,
        "remarks": ""
    }

In [0]:
def duplicate_check(spark, table_name, key_cols):
    df = spark.table(table_name)

    dup_count = df.groupBy(key_cols).count().filter("count > 1").count()

    return {
        "check_name": "duplicate_check",
        "status": "PASS" if dup_count == 0 else "FAIL",
        "src_count": df.count(),
        "tgt_count": None,
        "mismatch_count": None,
        "duplicate_count": dup_count,
        "null_count": None,
        "remarks": f"Checked on {key_cols}"
    }

In [0]:
def groupby_check(spark, src_table, tgt_table, group_cols):
    src_df = spark.table(src_table)
    tgt_df = spark.table(tgt_table)

    src_group = src_df.groupBy(group_cols).count()
    tgt_group = tgt_df.groupBy(group_cols).count()

    mismatch = src_group.subtract(tgt_group).count() + tgt_group.subtract(src_group).count()

    return {
        "check_name": "groupby_check",
        "status": "PASS" if mismatch == 0 else "FAIL",
        "src_count": src_df.count(),
        "tgt_count": tgt_df.count(),
        "mismatch_count": mismatch,
        "duplicate_count": None,
        "null_count": None,
        "remarks": f"grouped by {group_cols}"
    }

In [0]:
def null_check(spark, table_name):
    df = spark.table(table_name)
    cols = df.columns

    # Dynamic SQL
    null_expr = ", ".join([
        f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}`"
        for c in cols
    ])

    query = f"SELECT {null_expr} FROM {table_name}"

    result = spark.sql(query).collect()[0].asDict()

    # Pure Python sum (IMPORTANT)
    total_nulls = sum(v for v in result.values() if v is not None)

    return {
        "check_name": "null_check",
        "status": "PASS" if total_nulls == 0 else "FAIL",
        "src_count": df.count(),
        "tgt_count": None,
        "mismatch_count": None,
        "duplicate_count": None,
        "null_count": total_nulls,
        "remarks": "SQL-based null check"
    }

In [0]:
def referential_check(spark, child_table, parent_table, join_col):
    child_df = spark.table(child_table)
    parent_df = spark.table(parent_table)

    invalid_df = child_df.join(parent_df, join_col, "left_anti")
    invalid_count = invalid_df.count()

    return {
        "check_name": "referential_check",
        "status": "PASS" if invalid_count == 0 else "FAIL",
        "src_count": child_df.count(),
        "tgt_count": parent_df.count(),
        "mismatch_count": invalid_count,
        "duplicate_count": None,
        "null_count": None,
        "remarks": f"{join_col} integrity check"
    }

In [0]:
def log_results(spark, results):
    from datetime import datetime as dt
    from pyspark.sql import Row
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType
    from pyspark.sql.functions import col, to_timestamp

    schema = StructType([
        StructField("check_name",      StringType(),   True),
        StructField("status",          StringType(),   True),
        StructField("src_count",       IntegerType(),  True),  
        StructField("tgt_count",       IntegerType(),  True),  
        StructField("mismatch_count",  IntegerType(),  True),  
        StructField("duplicate_count", IntegerType(),  True),  
        StructField("null_count",      IntegerType(),  True), 
        StructField("remarks",         StringType(),   True),
        StructField("run_id",          StringType(),   True),
        StructField("timestamp",       TimestampType(), True),
    ])

    current_user = spark.sql("SELECT current_user()").collect()[0][0]
    run_id = f"{current_user}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    standardized_rows = []

    for r in results:
        standardized_rows.append(Row(
            check_name = r.get("check_name"),
            status = r.get("status"),
            src_count = r.get("src_count") or 0,
            tgt_count = r.get("tgt_count") or 0,
            mismatch_count = r.get("mismatch_count") or 0,
            duplicate_count = r.get("duplicate_count") or 0,
            null_count = r.get("null_count") or 0,
            remarks = r.get("remarks") or "",
            run_id = run_id,
            timestamp = dt.now()
        ))

    df = spark.createDataFrame(standardized_rows, schema=schema)
    df.write.mode("append").saveAsTable("project_training_amh.default.qa_validation_results")